In [21]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

# =================================================================
# PASO 1: Cargar los Datos Ya Preprocesados
# =================================================================
df = pd.read_csv("data/partidos_entrenamiento.csv", parse_dates=["date"])

# =================================================================
# PASO 2: Definir las Características (X) y el Objetivo (y)
# =================================================================
# Incluimos de forma estricta los nuevos predictores que calcula tu script
features = [
    "goles_totales_marcados_local_10",
    "goles_totales_concedidos_local_10",
    "forma_puntos_local_10",
    "goles_totales_marcados_visitante_10",
    "goles_totales_concedidos_visitante_10",
    "forma_puntos_visitante_10",
    "dif_goles_marcados_10",
    "dif_goles_concedidos_10",
    "dif_forma_puntos_10",
    "puntos_fifa_local",
    "puntos_fifa_visitante",
    "dif_puntos_fifa",
    "ranking_pos_local",
    "ranking_pos_visitante",
    "importancia_torneo",
]

df = df[df["date"] < "2026-05-01"]

df_train = df[(df["date"] < "2023-06-01") & (df["neutral"] == True)].copy()
X_train = df_train[features]
y_train = df_train["resultado"]
df_test = df[(df["date"] >= "2023-06-01") & (df["neutral"] == True)].copy()
X_test = df_test[features]
y_test = df_test["resultado"]
y_test


modelo_gb = HistGradientBoostingClassifier(
    max_iter=200,  # reducimos arboles
    learning_rate=0.05,  # pasos más cortos
    class_weight="balanced",
    max_depth=4,  # limitamos la profundidad
    min_samples_leaf=50,  # Exigimos que al menos 30 partidos cumplan una regla para darla por válida
    random_state=42,
)

# =================================================================
# PASO 2: Entrenar el modelo (Aprender de los datos de entrenamiento)
# =================================================================
# Ajustamos el modelo usando las pistas (X_train) y las respuestas reales (y_train)
modelo_gb.fit(X_train, y_train)

# =================================================================
# PASO 3: Realizar las predicciones
# =================================================================
# Le pedimos al modelo que intente adivinar los resultados del 20% de partidos reservados
predicciones = modelo_gb.predict(X_test)

# =================================================================
# PASO 4: Evaluar los resultados en consola
# =================================================================
# Calculamos el porcentaje total de aciertos
precision = accuracy_score(y_test, predicciones)

print(f"La precisión es: {precision * 100:.2f}%")
print("\nPor cada resultado (1=Local, 0=Empate, 2=Visitante):")
print(classification_report(y_test, predicciones))


La precisión es: 53.41%

Por cada resultado (1=Local, 0=Empate, 2=Visitante):
              precision    recall  f1-score   support

         0.0       0.32      0.31      0.32       281
         1.0       0.65      0.54      0.59       438
         2.0       0.57      0.69      0.62       380

    accuracy                           0.53      1099
   macro avg       0.51      0.51      0.51      1099
weighted avg       0.54      0.53      0.53      1099



In [22]:
import pandas as pd
import numpy as np

# 1. Cargamos el archivo limpio que generó tu script de preprocesamiento
df_entrenamiento = pd.read_csv('data/partidos_entrenamiento.csv', parse_dates=['date'])

# 2. Creamos un diccionario vacío para guardar el estado de cada país
diccionario_maestro = {}

# 3. Agrupamos todos los equipos que existen en la base de datos (tanto locales como visitantes)
todas_las_selecciones = set(df_entrenamiento['home_team'].unique()).union(set(df_entrenamiento['away_team'].unique()))

for seleccion in todas_las_selecciones:
    # Buscamos los partidos donde participó esta selección
    partidos_equipo = df_entrenamiento[(df_entrenamiento['home_team'] == seleccion) | (df_entrenamiento['away_team'] == seleccion)]
    
    if not partidos_equipo.empty:
        # Tomamos el último partido de su historia (el más reciente en la línea de tiempo)
        ultimo_partido = partidos_equipo.sort_values('date').iloc[-1]
        
        # Extraemos sus métricas dependiendo de si en ese último partido jugó como local o visitante
        if ultimo_partido['home_team'] == seleccion:
            goles_totales_marcados_10 = ultimo_partido['goles_totales_marcados_local_10']
            goles_totales_concedidos_10 = ultimo_partido['goles_totales_concedidos_local_10']
            forma_puntos_10 = ultimo_partido['forma_puntos_local_10']
            puntos_fifa = ultimo_partido['puntos_fifa_local']
            ranking_pos = ultimo_partido['ranking_pos_local']
        else:
            goles_totales_marcados_10 = ultimo_partido['goles_totales_marcados_visitante_10']
            goles_totales_concedidos_10 = ultimo_partido['goles_totales_concedidos_visitante_10']
            forma_puntos_10 = ultimo_partido['forma_puntos_visitante_10']
            puntos_fifa = ultimo_partido['puntos_fifa_visitante']
            ranking_pos = ultimo_partido['ranking_pos_visitante']
            
        # Guardamos su "ficha técnica" en el diccionario maestro
        diccionario_maestro[seleccion] = {
            'goles_recientes': goles_totales_marcados_10,
            'goles_concedidos': goles_totales_concedidos_10,
            'forma_reciente': forma_puntos_10,
            'puntos_fifa': puntos_fifa,
            'ranking_fifa': ranking_pos
        }


In [23]:
def predecir_encuentro_mundial(local, visitante):
    # 1. Verificar si ambas selecciones existen en el diccionario maestro
    if local not in diccionario_maestro or visitante not in diccionario_maestro:
        return f"Error: Uno o ambos equipos ('{local}' / '{visitante}') no tienen suficiente historial en el CSV."
        
    # 2. Extraer los datos guardados de cada país de forma automatizada
    loc = diccionario_maestro[local]
    vis = diccionario_maestro[visitante]
    
    # 3. Construir la fila de 11 predictores calculando las diferencias (Locales menos Visitantes)
    datos_partido = {
        'importancia_torneo': 4.0, # Nivel de máxima importancia (Mundial)     
        'goles_totales_marcados_local_10': loc['goles_recientes'],
        'goles_totales_concedidos_local_10': loc['goles_concedidos'],
        'forma_puntos_local_10' : loc['forma_reciente'],
        'puntos_fifa_local' : loc ['puntos_fifa'],
        'ranking_pos_local' : loc ['ranking_fifa'],
        ##visitante
        'goles_totales_marcados_visitante_10': vis['goles_recientes'],
        'goles_totales_concedidos_visitante_10': vis['goles_concedidos'],
        'forma_puntos_visitante_10' : vis['forma_reciente'],
        'puntos_fifa_visitante' : vis ['puntos_fifa'],
        'ranking_pos_visitante' : vis ['ranking_fifa'],
        # Calculamos los predictores diferenciales automáticos
        'dif_goles_marcados_10': loc['goles_recientes'] - vis['goles_recientes'],
        'dif_forma_puntos_10': loc['forma_reciente'] - vis['forma_reciente'],
        'dif_goles_concedidos_10': loc['goles_concedidos'] - vis['goles_concedidos'],
        'dif_puntos_fifa': loc['puntos_fifa'] - vis['puntos_fifa'],
        'dif_ranking_fifa': loc['ranking_fifa'] - vis['ranking_fifa']
    }
    
    # Convertimos en el DataFrame estructurado que exige tu HistGradientBoosting
    X_partido = pd.DataFrame([datos_partido])
    
    # 4. Extraer las probabilidades asignadas por tu modelo entrenado
    probabilidades = modelo_gb.predict_proba(X_partido[features])[0]
    
    p_empate = probabilidades[0]
    p_local = probabilidades[1]
    p_visitante = probabilidades[2]

    print(f"Probabilidad de Victoria de {local}: {p_local*100:.2f}%")
    print(f"Probabilidad de Empate: {p_empate*100:.2f}%")
    print(f"Probabilidad de Victoria de {visitante}: {p_visitante*100:.2f}%\n")
    
    return {'local': p_local, 'empate': p_empate, 'visitante': p_visitante}

# Escribe cualquier partido que aparezca en el fixture (Respeta las mayúsculas del CSV original)
_ = predecir_encuentro_mundial('Portugal', 'Spain')



Probabilidad de Victoria de Portugal: 25.77%
Probabilidad de Empate: 40.46%
Probabilidad de Victoria de Spain: 33.77%



In [24]:
import joblib

# Guardamos el modelo entrenado en un archivo físico
joblib.dump(modelo_gb, "modelo_gb.pkl")
# Para cargarlo en el futuro, simplemente usamos:
# modelo_cargado = joblib.load("modelo_gb.pkl")

['modelo_gb.pkl']

In [25]:
import pandas as pd
import numpy as np
import joblib

# =====================================================================
# PASO 1: CONFIGURACIÓN INICIAL Y CARGA DE ELEMENTOS
# =====================================================================
# Asegúrate de haber guardado tu modelo en el script anterior usando joblib.dump(modelo_gb, "modelo_gb.pkl")
modelo_gb = joblib.load("modelo_gb.pkl")

# Cargamos el calendario oficial del Mundial 2026
df_mundial = pd.read_csv('data/partidos_entrenamiento.csv')

# Tus features exactas de entrenamiento
features = [
    "goles_totales_marcados_local_10", "goles_totales_concedidos_local_10", "forma_puntos_local_10",
    "goles_totales_marcados_visitante_10", "goles_totales_concedidos_visitante_10", "forma_puntos_visitante_10",
    "dif_goles_marcados_10", "dif_goles_concedidos_10", "dif_forma_puntos_10",
    "puntos_fifa_local", "puntos_fifa_visitante", "dif_puntos_fifa",
    "ranking_pos_local", "ranking_pos_visitante"
]

# =====================================================================
# PASO 2: TU FUNCIÓN MEJORADA CON DATA AUGMENTATION (SIMETRÍA PERFECTA)
# =====================================================================
def predecir_encuentro_mundial(local, visitante, mostrar_consola=False):
    if local not in diccionario_maestro or visitante not in diccionario_maestro:
        return None
        
    loc = diccionario_maestro[local]
    vis = diccionario_maestro[visitante]
    
    # Escenario A: Local vs Visitante
    datos_a = {
        'goles_totales_marcados_local_10': loc['goles_recientes'],
        'goles_totales_concedidos_local_10': loc['goles_concedidos'],
        'forma_puntos_local_10' : loc['forma_reciente'],
        'puntos_fifa_local' : loc['puntos_fifa'],
        'ranking_pos_local' : loc['ranking_fifa'],
        'goles_totales_marcados_visitante_10': vis['goles_recientes'],
        'goles_totales_concedidos_visitante_10': vis['goles_concedidos'],
        'forma_puntos_visitante_10' : vis['forma_reciente'],
        'puntos_fifa_visitante' : vis['puntos_fifa'],
        'ranking_pos_visitante' : vis['ranking_fifa'],
        'dif_goles_marcados_10': loc['goles_recientes'] - vis['goles_recientes'],
        'dif_goles_concedidos_10': loc['goles_concedidos'] - vis['goles_concedidos'],
        'dif_forma_puntos_10': loc['forma_reciente'] - vis['forma_reciente'],
        'dif_puntos_fifa': loc['puntos_fifa'] - vis['puntos_fifa'],
        'dif_ranking_pos': loc['ranking_fifa'] - vis['ranking_fifa'] # Ajustado nombre para coincidir con tus features
    }
    
    # Escenario B: Inversión de roles completa (Visitante pasa a ser Local administrativo)
    datos_b = {
        'goles_totales_marcados_local_10': vis['goles_recientes'],
        'goles_totales_concedidos_local_10': vis['goles_concedidos'],
        'forma_puntos_local_10' : vis['forma_reciente'],
        'puntos_fifa_local' : vis['puntos_fifa'],
        'ranking_pos_local' : vis['ranking_fifa'],
        'goles_totales_marcados_visitante_10': loc['goles_recientes'],
        'goles_totales_concedidos_visitante_10': loc['goles_concedidos'],
        'forma_puntos_visitante_10' : loc['forma_reciente'],
        'puntos_fifa_visitante' : loc['puntos_fifa'],
        'ranking_pos_visitante' : loc['ranking_fifa'],
        'dif_goles_marcados_10': vis['goles_recientes'] - loc['goles_recientes'],
        'dif_goles_concedidos_10': vis['goles_concedidos'] - loc['goles_concedidos'],
        'dif_forma_puntos_10': vis['forma_reciente'] - loc['forma_reciente'],
        'dif_puntos_fifa': vis['puntos_fifa'] - loc['puntos_fifa'],
        'dif_ranking_pos': vis['ranking_fifa'] - loc['ranking_fifa']
    }
    
    # Evaluamos en el modelo de forma independiente
    prob_a = modelo_gb.predict_proba(pd.DataFrame([datos_a])[features])[0]
    prob_b = modelo_gb.predict_proba(pd.DataFrame([datos_b])[features])[0]
    
    # Promediamos alineando las etiquetas (el empate columna 0 queda igual, invertimos 1 y 2 en B)
    p_empate = (prob_a[0] + prob_b[0]) / 2
    p_local = (prob_a[1] + prob_b[2]) / 2
    p_visitante = (prob_a[2] + prob_b[1]) / 2
    
    if mostrar_consola:
        print(f"Probabilidad de Victoria de {local}: {p_local*100:.2f}%")
        print(f"Probabilidad de Empate: {p_empate*100:.2f}%")
        print(f"Probabilidad de Victoria de {visitante}: {p_visitante*100:.2f}%\n")
        
    return {'empate': p_empate, 'local': p_local, 'visitante': p_visitante}

# =====================================================================
# PASO 3: FUNCIÓN DEL DADO ALEATORIO DE MONTECARLO
# =====================================================================
def resolver_partido_montecarlo(p_empate, p_local, p_visitante):
    dado = np.random.rand()
    if dado < p_empate:
        return 0  # Empate (1 punto para cada uno)
    elif dado < (p_empate + p_local):
        return 1  # Gana el de la izquierda (Local administrativo)
    else:
        return 2  # Gana el de la derecha (Visitante administrativo)

# =====================================================================
# PASO 4: EJECUCIÓN DEL BUCLE MAESTRO DE MONTECARLO
# =====================================================================
N_SIMULACIONES = 5000
conteo_clasificados = {}

# CORRECCIÓN: Usamos ['group'] con corchetes para evitar el conflicto con métodos de Python
dict_grupos = pd.Series(df_mundial['group'].values, index=df_mundial.home_team).to_dict()
dict_grupos.update(pd.Series(df_mundial['group'].values, index=df_mundial.away_team).to_dict())

print(f"🎰 Ejecutando {N_SIMULACIONES} simulaciones de Montecarlo basadas en tu diccionario maestro...")

for sim in range(N_SIMULACIONES):
    # Inicializamos todos los puntos de este Mundial concreto en 0
    puntos_sim = {equipo: 0 for equipo in dict_grupos.keys()}
    
    # Recorremos el fixture partido por partido
    for idx, partido in df_mundial.iterrows():
        loc = partido['home_team']
        vis = partido['away_team']
        
        # Obtenemos las probabilidades simétricas usando tu función integrada
        probs = predecir_encuentro_mundial(loc, vis)
        
        if probs is None:
            continue # Control por si alguna selección del fixture no está en el diccionario maestro
            
        # Lanzamos el dado de Montecarlo
        resultado = resolver_partido_montecarlo(probs['empate'], probs['local'], probs['visitante'])
        
        if resultado == 1:
            puntos_sim[loc] += 3
        elif resultado == 2:
            puntos_sim[vis] += 3
        else:
            puntos_sim[loc] += 1
            puntos_sim[vis] += 1
            
    # --- PROCESAR POSICIONES DE LOS GRUPOS EN ESTA SIMULACIÓN ---
    df_puntos_sim = pd.DataFrame(list(puntos_sim.items()), columns=['equipo', 'puntos'])
    df_puntos_sim['grupo'] = df_puntos_sim['equipo'].map(dict_grupos)
    
    # Ordenamos de forma descendente por puntos dentro de cada grupo
    df_puntos_sim = df_puntos_sim.sort_values(by=['grupo', 'puntos'], ascending=[True, False])
    
    # Extraemos las dos mejores selecciones (.head(2)) de cada grupo
    clasificados_fase_grupos = df_puntos_sim.groupby('grupo').head(2)['equipo'].values
    
    # Registramos el éxito en nuestro contador general
    for equipo in clasificados_fase_grupos:
        conteo_clasificados[equipo] = conteo_clasificados.get(equipo, 0) + 1

# =====================================================================
# PASO 5: PRESENTACIÓN DE LOS RESULTADOS MATEMÁTICOS DE CLASIFICACIÓN
# =====================================================================
df_resultados_finales = pd.DataFrame([
    {"Selección": eq, "Grupo": dict_grupos[eq], "Probabilidad_Clasificar_%": (votos / N_SIMULACIONES) * 100}
    for eq, votos in conteo_clasificados.items()
])

# Ordenamos el output estético por grupo y de mayor a menor probabilidad
df_resultados_finales = df_resultados_finales.sort_values(by=["Grupo", "Probabilidad_Clasificar_%"], ascending=[True, False])

print("\n🏆 RESULTADOS FINALES MONTECARLO: PROBABILIDAD DE PASAR A LA SIGUIENTE RONDA")
print("=================================================================================")
print(df_resultados_finales.to_string(index=False))

KeyError: 'group'